Parte previa, git-github

1. Flujo de Trabajo en Git (MLOps Básico)
Primero, inicializar repositorio y gestionar las ramas como pide la consigna:

Paso 1: Configuración Inicial de Git y GitHub

Bash
git config --global user.name "Tu Nombre" en este casó usé mi cuenta de gmail donde también tengo colab pro
git config --global user.email "tu@email.com"
Luego creé una cuenta de github.

Paso 2: Inicializar el Proyecto (Rama master)
Convertimos la carpeta actual en un repositorio de Git.

En la terminal, dentro de tu carpeta:

Bash
git init
# Crear una rama principal limpia
git checkout -b master
Conectar con GitHub: Copiamos la URL del repositorio de GitHub 

git remote add origin https://github.com/screamingvalkirias-png/tarea-redes-neuronales.git

Tuve que pegar esto cd "3er Trimestre/Redes Neuronales/TAREA1" en el bash para ubicar la tarea. Luego cambio la rama etapa1:
git checkout -b etapa1
Luego
git push origin etapa1


Parte de encoder

Paso 1: Carga y Preprocesamiento de Datos

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

# 1. Cargar el dataset (no necesitamos las etiquetas 'y' por ahora)
(x_train, _), (x_test, _) = tf.keras.datasets.fashion_mnist.load_data()

# 2. Normalización: Pasamos los píxeles de [0, 255] a [0, 1]
# Esto ayuda a que la red neuronal converja más rápido.
x_train = x_train.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.

print(f"Forma de los datos: {x_train.shape}") # Debería ser (60000, 28, 28)

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Forma de los datos: (60000, 28, 28)


Paso 2: Arquitectura del Autoencoder
Divido la red en dos partes. El Encoder volveremos a usar en la Etapa 2 de la tarea.

In [2]:
# Definir la dimensión del vector latente (la compresión)
# Un valor de 64 significa que comprimiremos 784 píxeles en solo 64 valores.
latent_dim = 64 

# --- ENCODER ---
encoder_input = layers.Input(shape=(28, 28))
x = layers.Flatten()(encoder_input)
latent_vector = layers.Dense(latent_dim, activation='relu', name="vector_latente")(x)
encoder = models.Model(encoder_input, latent_vector, name="Encoder")

# --- DECODER ---
decoder_input = layers.Input(shape=(latent_dim,))
x = layers.Dense(784, activation='sigmoid')(decoder_input)
decoder_output = layers.Reshape((28, 28))(x)
decoder = models.Model(decoder_input, decoder_output, name="Decoder")

# --- AUTOENCODER COMPLETO ---
autoencoder_output = decoder(encoder(encoder_input))
autoencoder = models.Model(encoder_input, autoencoder_output, name="Autoencoder_Completo")

autoencoder.compile(optimizer='adam', loss='mean_squared_error')
autoencoder.summary()

Model: "Autoencoder_Completo"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Encoder (Functional)            │ (None, 64)             │        50,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Decoder (Functional)            │ (None, 28, 28)         │        50,960 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,200 (395.31 KB)

 Trainable params: 101,200 (395.31 KB)

 Non-trainable params: 0 (0.00 B)

Paso 3: Entrenamiento y Guardado
Entreno el modelo para que la salida sea lo más parecida posible a la entrada.

In [3]:
# Entrenamos: fíjate que x_train es tanto la entrada como la salida esperada
history = autoencoder.fit(
    x_train, x_train,
    epochs=10,
    batch_size=256,
    shuffle=True,
    validation_data=(x_test, x_test)
)

# GUARDAR EL ENCODER: Esto es vital para la Etapa 2
# Guardamos solo la parte que comprime los datos.
encoder.save('encoder_fashion_mnist.h5')
print("¡Encoder guardado con éxito!")



Epoch 1/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 0.0831 - val_loss: 0.0283
Epoch 2/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.0255 - val_loss: 0.0195
Epoch 3/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0184 - val_loss: 0.0158
Epoch 4/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0152 - val_loss: 0.0139
Epoch 5/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0135 - val_loss: 0.0127
Epoch 6/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 0.0124 - val_loss: 0.0119
Epoch 7/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 0.0117 - val_loss: 0.0113
Epoch 8/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.0111 - val_loss: 0.0109
Epoch 9/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0107 - val_loss: 0.0106
Epoch 10/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0104 - val_loss: 0.0103


¡Encoder guardado con éxito!


Paso 4: MLOps - Guardar cambios en Git
Busco que el código corra y ver el loss bajar, ahí se registra el avance en la rama etapa1:

En la terminal ejecuto:

git add PARTE1.ipynb encoder_fashion_mnist.h5

git commit -m "Etapa 1: Autoencoder entrenado y encoder exportado"

git push origin etapa1